# 15｜電腦視覺_遷移學習與偵測分割

用 IoU 與 NMS 小例子連接偵測框、信心分數與後處理。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/15_電腦視覺_遷移學習與偵測分割.md`，目前 30 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

IoU/NMS 手算、逐框配對與 AP、分割/上採樣、轉置線性算子、追蹤。

各格會標示實驗資料、評估方式與適用範圍；架構圖用於說明機制與張量形狀。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 15-03 | 12、17、20 | 課堂小實驗 |
| 15-04 | 3、12、17 | 執行結果圖 |
| 15-05 | 12、13、16、17、18、19、20、30 | 1. 相同手算、逐框配對與 AP |
| 15-06 | 4、11、14、15、22、23、24、25、26、27、30 | 2. 全卷積輸出、轉置卷積與跳接 |
| 15-07 | 5、6、7、8、9、10、21、27、28、29、30 | 3. 前處理、凍結頭與追蹤配對 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 15-01｜投影片 15：2 頁
import importlib.util, subprocess, sys
_needed = {'scipy': 'scipy>=1.11,<2', 'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 15-02｜投影片 15：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 課堂小實驗

目前候選皆視為同類，NMS 的抑制條件是 IoU ≥ 0.5；正式多類偵測要按類別處理。

In [ ]:
# 程式 15-03｜投影片 15：12、17、20 頁
boxes = np.array([[0.10,0.10,0.55,0.55], [0.16,0.14,0.58,0.57], [0.62,0.58,0.92,0.90]])
scores = np.array([0.92, 0.81, 0.74])
def iou(a, b):
    lo = np.maximum(a[:2], b[:2]); hi = np.minimum(a[2:], b[2:])
    inter = np.prod(np.maximum(hi - lo, 0))
    area_a = np.prod(a[2:] - a[:2]); area_b = np.prod(b[2:] - b[:2])
    return inter / (area_a + area_b - inter)
keep = []
for idx in np.argsort(scores)[::-1]:
    if all(iou(boxes[idx], boxes[j]) < 0.5 for j in keep): keep.append(int(idx))
table = pd.DataFrame(boxes, columns=['x1','y1','x2','y2']).assign(score=scores, kept=[i in keep for i in range(3)])
print('IoU(box 0, box 1)=', round(iou(boxes[0], boxes[1]), 3))
table

## 執行結果圖

In [ ]:
# 程式 15-04｜投影片 15：3、12、17 頁
from matplotlib.patches import Rectangle
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, ids, title in [(axes[0], range(3), 'before NMS'), (axes[1], keep, 'after NMS')]:
    for i in ids:
        x1,y1,x2,y2 = boxes[i]
        ax.add_patch(Rectangle((x1,y1), x2-x1, y2-y1, fill=False, lw=3, label=f'{scores[i]:.2f}'))
    ax.set(xlim=(0,1), ylim=(1,0), title=title); ax.legend()
savefig('15_detection_demo')
report('vision_detection', {'keep': keep, 'iou_0_1': iou(boxes[0], boxes[1])})

## 1. 相同手算、逐框配對與 AP

一對一配對避免同一真物件的重複框反覆算 TP。以下示範單類、單影像、IoU=0.5 的全點插值 AP，不是 COCO mAP。

In [ ]:
# 程式 15-05｜投影片 15：12、13、16、17、18、19、20、30 頁
assert np.isclose(60/(100+100-60),3/7)
hand_iou15=np.array([[1,.8,.1],[.8,1,.1],[.1,.1,1]])
kept15=[]
for j in [0,1,2]:
    if all(hand_iou15[j,k]<.5 for k in kept15):kept15.append(j)
assert kept15==[0,2]
gt15=np.array([[.1,.1,.55,.55],[.62,.58,.92,.90]])
matched15=set();tp15=[]
for j in np.argsort(scores)[::-1]:
    overlaps15=np.array([iou(boxes[j],g) for g in gt15]);order15=np.argsort(overlaps15)[::-1]
    hit15=next((k for k in order15 if k not in matched15 and overlaps15[k]>=.5),None)
    tp15.append(hit15 is not None)
    if hit15 is not None:matched15.add(hit15)
recall15=np.cumsum(tp15)/len(gt15);precision15=np.cumsum(tp15)/np.arange(1,len(tp15)+1)
r15=np.r_[0,recall15,1];p15=np.r_[0,precision15,0];envelope15=np.maximum.accumulate(p15[::-1])[::-1]
ap15=np.sum(np.diff(r15)*envelope15[1:])
assert tp15==[True,False,True] and np.isclose(ap15,5/6)
plt.step(recall15,precision15,where='post',label='raw PR');plt.step(r15,envelope15,where='post',label='precision envelope');plt.legend();plt.xlabel('recall');plt.ylabel('precision');savefig('c15_detection_ap')
print('TP flags / interpolated AP@.5:',tp15,ap15)

## 2. 全卷積輸出、轉置卷積與跳接

以二維玩具資料驗算空間輸出；轉置卷積是線性算子的轉置，不是卷積的逆。像素標籤、實例 ID 與全景 ID 分開保存。

In [ ]:
# 程式 15-06｜投影片 15：4、11、14、15、22、23、24、25、26、27、30 頁
from numpy.lib.stride_tricks import sliding_window_view
scene15=np.zeros((12,16),int);scene15[2:7,2:6]=1;scene15[4:10,9:14]=1
instances15=np.zeros_like(scene15);instances15[2:7,2:6]=1;instances15[4:10,9:14]=2
# 每位置共享線性分類器（1×1 conv）：不同空間大小均可前向
rgb15=np.stack([scene15,scene15*.5,np.ones_like(scene15)*.1],axis=0)
w15=np.array([[-1.,0,0],[1.,0,0]]);b15=np.array([.5,-.5])
logits15=np.einsum('kc,chw->khw',w15,rgb15)+b15[:,None,None];semantic15=logits15.argmax(0)
assert np.array_equal(semantic15,scene15)
# 滑動線性視窗與全圖互相關等價
kernel15=np.array([[1.,0.],[0.,-1.]])
patches15=sliding_window_view(scene15,(2,2));dense15=np.einsum('hwij,ij->hw',patches15,kernel15)
assert dense15.shape==(11,15)
# 轉置卷積 scatter-add，stride=2
low15=np.array([[1.,2.],[3.,4.]]);kt15=np.ones((3,3));trans15=np.zeros((5,5))
for i in range(2):
    for j in range(2):trans15[i*2:i*2+3,j*2:j*2+3]+=low15[i,j]*kt15
# 驗證 <Cx,y>=<x,C^T y>，原卷積 stride=2
probe15=np.arange(25).reshape(5,5)/25
corr15=np.einsum('hwij,ij->hw',sliding_window_view(probe15,(3,3))[::2,::2],kt15)
assert np.isclose(np.sum(corr15*low15),np.sum(probe15*trans15))
coarse15=scene15.reshape(6,2,8,2).mean(axis=(1,3));up15=np.repeat(np.repeat(coarse15,2,0),2,1)
skip15=np.stack([up15,scene15],axis=0);assert skip15.shape==(2,12,16)
fig,axes=plt.subplots(1,4,figsize=(12,3))
for ax,arr,title in zip(axes,[semantic15,instances15,up15,trans15],['semantic class','instance IDs','upsample, before skip','transpose convolution']):ax.imshow(arr,cmap='viridis');ax.set_title(title);ax.axis('off')
savefig('c15_segmentation_upsampling')
intersection15=np.sum((up15>.5)&(scene15==1));union15=np.sum((up15>.5)|(scene15==1))
print('Toy foreground IoU before skip:',intersection15/union15,'panoptic (class,instance):',list(zip(scene15[5],instances15[5])))

## 3. 前處理、凍結頭與追蹤配對

合成固定特徵代表已學好的 encoder；只更新分類頭並檢查主幹不變。這不等於完整預訓練模型。追蹤配對須保留 ID，類別正確仍可能身分切換。

In [ ]:
# 程式 15-07｜投影片 15：5、6、7、8、9、10、21、27、28、29、30 頁
rng15=np.random.default_rng(SEED);images15=rng15.uniform(0,1,(20,3,8,8))
mean15=np.array([.5,.5,.5])[None,:,None,None];std15=np.array([.25,.25,.25])[None,:,None,None]
normalized15=(images15-mean15)/std15  # 合成前處理常數，不是指定預訓練權重的契約
backbone15=rng15.normal(size=(3,6));saved15=backbone15.copy();features15=normalized15.mean((2,3))@backbone15
head15=np.zeros((6,2));labels15=np.arange(20)%2
for _ in range(30):
    z15=features15@head15;z15-=z15.max(1,keepdims=True);p15=np.exp(z15);p15/=p15.sum(1,keepdims=True);p15[np.arange(20),labels15]-=1
    head15-=.1*features15.T@p15/20
assert np.array_equal(backbone15,saved15)
print('Backbone/head parameters:',backbone15.size,head15.size,'FP32 weights bytes:',4*(backbone15.size+head15.size))
previous15=np.array([[0,0,2,2],[4,0,6,2]],float);next15=np.array([[4.2,0,6.2,2],[.2,0,2.2,2]])
overlap15=np.array([[iou(a,b) for b in next15] for a in previous15])
from scipy.optimize import linear_sum_assignment
row15,col15=linear_sum_assignment(-overlap15)
assert col15.tolist()==[1,0]
print('Previous object ID -> next detection index:',dict(zip(row15.tolist(),col15.tolist())))

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。